# 03b — Plant transfer check: how general are the 03 models?

The 03 models were validated with **grouped cross-validation**. That tests *new recipes
from plants the model has seen*. A general tool must also work at a plant it has **never
seen**. This notebook tests exactly that, with the data we have, and writes the result
to `plant_transfer.json`, which 08 reads.

Two questions:

1. **Leave one plant out.** Train on every other plant, predict the missing one. How far
   off is each plant on average? The spread of those offsets (τ, "tau") is the extra
   uncertainty a brand-new plant carries.
2. **PPC as a blend.** Only one plant in the data uses PPC. PPC is Portland clinker
   ground with fly ash (IS 1489 Part 1, 15–35 % fly ash). If PPC is fed to the model as
   *OPC + that fly ash*, can a model that has **never seen PPC** predict the PPC plant?

This notebook **does not change or retrain 03's saved models**. It re-uses 03's own
feature, grouping and model code — read straight out of `03_Modelling_audited.ipynb` —
so the method is identical.

In [13]:
import json, io, contextlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

from is_rules import IS1489_1_FLY_ASH_CONSTITUENT_PCT       # PPC fly ash range, IS 1489 (Part 1) Amd 3

DATA_FILE      = "01_RAW_dataset.csv"          # the raw data in this folder (.csv or .xlsx)
MODEL_NOTEBOOK = "03_Modelling_audited.ipynb"  # method is read from here, never copied by hand
PPC_X_GENERAL  = IS1489_1_FLY_ASH_CONSTITUENT_PCT[1]   # 35 %: the general basis used by 00
EXCLUDE_FROM_TAU = ("Plant B",)   # 03 section 8: Plant B holds every mix below w/b 0.27, so
                              # holding it out tests extrapolation, not transfer

## 1. Load 03's method and the data

In [14]:
nb3 = json.load(open(MODEL_NOTEBOOK))
cells3 = ["".join(c["source"]) for c in nb3["cells"] if c["cell_type"] == "code"]
pick = lambda key: next(s for s in cells3 if key in s)
SETTINGS = pick("GROUP_TOL")
FEAT     = pick("df['wb']")
GROUPS   = pick("def recipe_groups")
MODEL    = pick("def fit_predict")

G = {}
exec(SETTINGS.split("DATA_FILE")[0], G)            # imports only
exec(SETTINGS[SETTINGS.index("GROUP_TOL"):], G)    # GROUP_TOL, FLOOR_FRAC, N_SEEDS, RUN_AUDIT
exec(pick("TEXT_COLS").split("raw = pd.read_excel")[0], G)   # TEXT_COLS

if DATA_FILE.endswith(".csv"):
    raw = pd.read_csv(DATA_FILE, low_memory=False)
else:                                              # 03's own spreadsheet layout
    r = pd.read_excel(DATA_FILE, header=None); raw = r.iloc[3:].copy(); raw.columns = r.iloc[1].tolist()
df = raw[raw.record_id.notna()].reset_index(drop=True)
for c in df.columns:
    if c not in G["TEXT_COLS"]: df[c] = pd.to_numeric(df[c], errors="coerce")
# 03 section 1 cleaning, unchanged
for k in ["flyash", "ggbs", "other_scm"]: df[k] = df[k].fillna(0)
df["binder"] = df[["cement", "flyash", "ggbs", "other_scm"]].sum(axis=1)
df["coarse"] = df[["agg_20mm", "agg_10mm", "agg_40mm", "agg_other"]].fillna(0).sum(axis=1)
df["fine"]   = df[["crushed_sand", "natural_sand"]].fillna(0).sum(axis=1)
df["mass"]   = df.binder + df.water_batched + df.coarse + df.fine + df.admixture1.fillna(0)
keep = (df.coarse > 0) & (df.mass >= 2150) & (df.mass <= 2750)
BASE = df[keep].copy()
print(f"{DATA_FILE}: {len(df)} rows, {keep.sum()} kept by 03's cleaning")
print(BASE.groupby("plant_lab_id").size().rename("rows").to_string())


def build(ppc_x=None):
    """Recipe-group tables with 03's code. ppc_x=None: PPC as recorded (is_ppc = 1).
    ppc_x = x: every PPC row re-expressed as (1 - x) OPC + x fly ash, is_ppc = 0. [EI]"""
    d = BASE.copy()
    if ppc_x is not None:
        ppc = d.cement_type.astype(str).str.upper().str.contains("PPC")
        d.loc[ppc, "flyash"] = d.loc[ppc, "flyash"] + d.loc[ppc, "cement"] * ppc_x / 100
        d.loc[ppc, "cement"] = d.loc[ppc, "cement"] * (1 - ppc_x / 100)
        d.loc[ppc, "cement_type"] = "OPC-equivalent"
    G["df"] = d
    with contextlib.redirect_stdout(io.StringIO()):
        exec(FEAT, G); exec(GROUPS, G); exec(MODEL, G)
    return {"fc": G["DS"], "slump_mm": G["DW"]}

01_RAW_dataset.csv: 344 rows, 340 kept by 03's cleaning
plant_lab_id
Plant B                               38
P1                                 3
Plant D                       256
Plant A     18
Plant C                              25


## 2. Leave one plant out

For each plant: train on all the others, predict its recipes, and record the average
offset in log space (`log_bias` = measured − predicted; −0.10 ≈ the model is 10 % too high).

In [15]:
rows = []
for basis, x in (("PPC as recorded", None), (f"PPC as OPC + {PPC_X_GENERAL:.0f}% fly ash", PPC_X_GENERAL)):
    T = build(x)
    for tgt, D in T.items():
        for p in sorted(D.plant.unique()):
            te = np.where(D.plant == p)[0]; tr = np.setdiff1d(np.arange(len(D)), te)
            if len(te) < 2:
                continue
            mu, sd = G["fit_predict"](D, tgt, tr, te)
            y, yhat = np.exp(D.ly.values[te]), np.exp(mu)
            rows.append({"basis": basis, "target": tgt, "plant": p[:26], "groups": len(te),
                         "batches": int(D.n.values[te].sum()),
                         "log_bias": float(np.mean(D.ly.values[te] - mu)),
                         "rmse": float(np.sqrt(np.mean((y - yhat) ** 2))),
                         "model_sd_log": float(np.mean(sd))})
LOPO = pd.DataFrame(rows)
print(LOPO.round(3).to_string(index=False))

                   basis   target                      plant  groups  batches  log_bias   rmse  model_sd_log
         PPC as recorded       fc                        Plant B      35       38     0.084  8.858         0.143
         PPC as recorded       fc                         P1       2        3    -0.193  9.637         0.086
         PPC as recorded       fc                 Plant D       5      256    -0.081  2.910         0.535
         PPC as recorded       fc Plant A      15       18    -0.073  5.433         0.109
         PPC as recorded       fc                       Plant C      14       25     0.007  5.488         0.156
         PPC as recorded slump_mm                        Plant B      14       14     0.051 35.798         0.201
         PPC as recorded slump_mm                         P1       2        3    -0.448 70.423         0.138
         PPC as recorded slump_mm                 Plant D       5      256    -0.566 82.969         0.174
         PPC as recorded slump_mm

In [16]:
TAU = {}
for (basis, tgt), g in LOPO.groupby(["basis", "target"]):
    g2 = g[~g.plant.str.startswith(EXCLUDE_FROM_TAU)]
    tau = float(g2.log_bias.std(ddof=1))
    if basis.startswith("PPC as OPC"):
        TAU[tgt] = {"tau_log": tau, "n_plants": int(len(g2)),
                    "plants": g2.plant.tolist(), "model_sd_log": float(g2.model_sd_log.mean())}
    print(f"{basis:26s} {tgt:9s} tau (SD of plant offsets, log) = {tau:.3f} over {len(g2)} plants"
          f"  | model's own sd ~ {g2.model_sd_log.mean():.3f}")

PPC as OPC + 35% fly ash   fc        tau (SD of plant offsets, log) = 0.100 over 4 plants  | model's own sd ~ 0.112
PPC as OPC + 35% fly ash   slump_mm  tau (SD of plant offsets, log) = 0.430 over 4 plants  | model's own sd ~ 0.238
PPC as recorded            fc        tau (SD of plant offsets, log) = 0.082 over 4 plants  | model's own sd ~ 0.221
PPC as recorded            slump_mm  tau (SD of plant offsets, log) = 0.437 over 4 plants  | model's own sd ~ 0.176


## 3. PPC as a blend — can a model with no PPC predict the PPC plant?

Hold out every PPC plant. Train only on OPC plants. Predict the PPC recipes several ways.

In [17]:
rows = []
scen = [("as recorded (is_ppc = 1)", None), ("as plain OPC (x = 0)", 0.0)] + \
       [(f"OPC + {x:.0f}% fly ash", x) for x in (IS1489_1_FLY_ASH_CONSTITUENT_PCT[0], 25.0,
                                                  IS1489_1_FLY_ASH_CONSTITUENT_PCT[1])]
for label, x in scen:
    T = build(x)
    for tgt, D in T.items():
        ppc_plants = BASE[BASE.cement_type.astype(str).str.upper().str.contains("PPC")].plant_lab_id.astype(str).unique()
        te = np.where(D.plant.isin(ppc_plants))[0]; tr = np.setdiff1d(np.arange(len(D)), te)
        if len(te) == 0:
            continue
        mu, sd = G["fit_predict"](D, tgt, tr, te)
        y, yhat = np.exp(D.ly.values[te]), np.exp(mu)
        rows.append({"PPC fed to the model as": label, "target": tgt, "groups": len(te),
                     "batches": int(D.n.values[te].sum()),
                     "bias (pred - meas)": float(np.mean(yhat - y)),
                     "rmse": float(np.sqrt(np.mean((yhat - y) ** 2))),
                     "predicted": np.round(yhat, 1).tolist(), "measured": np.round(y, 1).tolist()})
PPC_TEST = pd.DataFrame(rows)
print(PPC_TEST.round(2).to_string(index=False))

 PPC fed to the model as   target  groups  batches  bias (pred - meas)  rmse                           predicted                           measured
as recorded (is_ppc = 1)       fc       5      256                2.63  2.91      [47.7, 37.3, 29.4, 33.3, 43.0]     [46.4, 35.7, 24.5, 30.4, 40.4]
as recorded (is_ppc = 1) slump_mm       5      256               82.54 82.97 [193.4, 191.0, 190.4, 190.8, 194.0] [121.6, 108.6, 94.8, 103.8, 118.1]
    as plain OPC (x = 0)       fc       5      256                1.55  1.99      [49.4, 36.0, 25.7, 30.7, 43.4]     [46.4, 35.7, 24.5, 30.4, 40.4]
    as plain OPC (x = 0) slump_mm       5      256               82.22 82.63 [193.0, 189.9, 190.2, 190.3, 194.6] [121.6, 108.6, 94.8, 103.8, 118.1]
       OPC + 15% fly ash       fc       5      256                0.64  1.38      [48.5, 35.1, 24.8, 29.8, 42.5]     [46.4, 35.7, 24.5, 30.4, 40.4]
       OPC + 15% fly ash slump_mm       5      256               83.79 84.16 [194.9, 191.5, 191.1, 191.5, 196.8]

## 4. Save what 08 needs

In [18]:
s = PPC_TEST[PPC_TEST.target == "fc"].set_index("PPC fed to the model as")
out = {
    "data_file": DATA_FILE,
    "ppc_model_x_pct": PPC_X_GENERAL,
    "tau": TAU,
    "ppc_strength_test": {k: {"bias": float(v["bias (pred - meas)"]), "rmse": float(v["rmse"]),
                              "groups": int(v["groups"])} for k, v in s.iterrows()},
    "note": ("tau = SD across plants of the leave-one-plant-out mean log offset "
             "(Plant B excluded, 03 section 8). Few plants: treat as an estimate."),
}
json.dump(out, open("plant_transfer.json", "w"), indent=1)
print(json.dumps({k: out[k] for k in ("ppc_model_x_pct", "tau")}, indent=1))
print("\nwrote plant_transfer.json")

{
 "ppc_model_x_pct": 35.0,
 "tau": {
  "fc": {
   "tau_log": 0.10037714288252247,
   "n_plants": 4,
   "plants": [
    "P1",
    "Plant D",
    "Plant A",
    "Plant C"
   ],
   "model_sd_log": 0.11219735428225251
  },
  "slump_mm": {
   "tau_log": 0.4302650021624612,
   "n_plants": 4,
   "plants": [
    "P1",
    "Plant D",
    "Plant A",
    "Plant C"
   ],
   "model_sd_log": 0.23759999003942517
  }
 }
}

wrote plant_transfer.json


## 5. What this means

Read the numbers above before this text; the text only says how to use them.

* **Strength transfers.** If the plant offsets are small (τ around 0.1 in log, about 10 %),
  one strength model can serve a new plant, with τ added to its uncertainty.
* **PPC as OPC + fly ash** is supported if the no-PPC model predicts the PPC plant with a
  small bias. 08 then feeds PPC to the model this way, at the same 35 % that 00 uses for the
  IS checks — so the model and the rules describe the same cement.
* **Workability does not transfer** if τ for slump is several times the model's own sd.
  Slump depends on things the data does not record (aggregate moisture, admixture
  chemistry, time from batching, temperature). That is not a modelling failure: IS 10262
  cl 5.8 itself says the water and/or admixture of Trial Mix No. 1 *shall be adjusted* if
  the measured workability differs. So at a new plant, slump is **set by Trial Mix 1**, and
  the first measured slump calibrates the model for that plant.
* **Limit.** Few plants means τ is an estimate, and one PPC plant is one test. Every new
  plant's data should be added here and this notebook re-run.